# JupyterHub on NRP

**Afternoon session · 12:45 – 1:25 PM** · [website version](https://training.nrp-nautilus.io/pearc26/5_jupyterhub.html) — run cells with **Shift+Enter**.


## ⚙️ Setup — run this first

Set your short username once; every command below uses `$NRP_USER`. The cell
also renders every manifest into **`my-yamls/`** with `<username>` already
filled in — wherever the website says *"replace `<username>`"*, it's already
done for you here.

> Terminal steps below don't share this variable — run the same
> `export NRP_USER=...` line in any terminal you open.
> Re-running this cell re-renders `my-yamls/` (overwriting any edits you made there).

**First time in one of these notebooks?** Click the **📌 pin icon** in the toolbar
above for a 30-second guided tour of how this notebook works.


In [ ]:
export NRP_USER=changeme   # ✏️ EDIT to your short name, then Shift+Enter
cd ~/pearc26/workspace
if [ "$NRP_USER" = changeme ]; then echo "⚠️  Edit NRP_USER above first, then re-run"; else
  mkdir -p my-yamls
  for f in yamls/*; do sed "s/<username>/$NRP_USER/g" "$f" > "my-yamls/$(basename "$f")"; done
  echo "✅ my-yamls/ rendered for $NRP_USER"
fi


**Afternoon session · 12:45 – 1:25 PM**

You've spent the morning *inside* a JupyterHub. This episode opens the hood: what JupyterHub actually is, how it runs on Kubernetes, why it has become the standard for reproducible classroom and research computing, and the features you'd lean on when running your own course — before the final episode, where you deploy one yourself.

> 📘 **Docs:** [JupyterHub service](https://nrp.ai/documentation/userdocs/jupyter/jupyterhub-service/) · [Deploy JupyterHub](https://nrp.ai/documentation/userdocs/jupyter/jupyterhub/) · [Scientific images](https://nrp.ai/documentation/userdocs/running/sci-img/) · [Z2JH (upstream)](https://z2jh.jupyter.org)


## What JupyterHub is

JupyterHub is a multi-user gateway to single-user Jupyter servers: users authenticate, the hub **spawns** an isolated JupyterLab server per user, and a proxy routes each browser to the right server. On Kubernetes ("Zero to JupyterHub", z2jh), those pieces map to:

- **hub pod** — authentication, user database, and the *spawner* that creates user pods.
- **proxy pod** — routes incoming traffic to the hub or to the correct user server.
- **user pods** — one JupyterLab server per active user, created on demand, culled when idle.
- **PVCs** — one home volume per user (plus one for the hub database).

<div class="image-row">
  <img src="https://training.nrp-nautilus.io/pearc26/images/jhub-1.png" alt="JupyterHub spawner profile page">
  <img src="https://training.nrp-nautilus.io/pearc26/images/jhub-2.png" alt="JupyterLab session on NRP">
</div>

Because each user server is just a **pod**, everything from this morning applies: images define the software stack, resource requests define CPU/RAM/GPU, tolerations and affinity steer students onto reserved nodes, PVCs make home directories persistent.


## Why this matters for education and research

- **Reproducibility** — every student gets the identical container image; "works on my machine" disappears.
- **Zero install** — a laptop with a browser is the only prerequisite (today's tutorial required nothing else).
- **Institutional login** — CILogon/OIDC means students use campus credentials; no account provisioning.
- **Fair sharing** — per-user CPU/RAM/GPU limits and idle culling keep one user from starving a class.
- **Scale on national CI** — the same hub that serves 5 researchers serves a 300-student course; NRP supplies the nodes.


## The hosted NRP JupyterHubs

NRP operates hosted hubs you can use without deploying anything — e.g. [jupyterhub-west.nrp-nautilus.io](https://jupyterhub-west.nrp-nautilus.io) with CILogon institutional login. After signing in you pick a **profile** (CPU/GPU size and image) and land in JupyterLab. Home directories are persistent PVCs (5 GB default); idle servers are culled about an hour after your browser disconnects.

The tutorial hub you're on ([jh-training.nrp-nautilus.io](https://jh-training.nrp-nautilus.io)) is the same architecture, plus tutorial extras: `kubectl`/`helm` preinstalled, the LLM token injected, and every spawn steered to the reserved A10 pool.


## Hands-on: features you'd use in a course

**1. The spawner profile list.** Go to the hub control panel (**File → Hub Control Panel**), stop your server, and look at the spawn page: each entry is a `profileList` item mapping a display name to an image and resource set. You'll write one of these yourself in the next episode.

**2. Query the NRP LLM from Jupyter AI.** As in the morning: the chat panel and `%%ai` magics are wired to the managed LLM per spawn — a course-wide AI assistant with no per-student API keys. In a notebook:

*(In a Python 3 notebook, not here:)*

```python
%load_ext jupyter_ai_magics
```


```text
%%ai openai-chat:minimax-m2
Give me three exam-style questions about Kubernetes pods.
```


**3. Distribute materials with nbgitpuller.** The "Launch the workspace" button on every page of this site is an [nbgitpuller](https://jupyterhub.github.io/nbgitpuller/) link: it signs the student in, clones/updates a git repo into their home directory **without ever overwriting their edits**, and opens a target path. Build your own links with the [nbgitpuller link generator](https://nbgitpuller.readthedocs.io/en/latest/link.html) — this is the standard way to hand out assignments.

**4. Real-time collaboration.** JupyterLab's collaborative mode (shared documents, live cursors) can be enabled per hub — useful for pair exercises and office hours.


## Sizing a hub for a course

Rules of thumb the NRP team uses when provisioning class hubs:

- **Concurrency, not enrollment** — a 100-student course rarely exceeds ~40 simultaneous servers outside deadline nights.
- **Right-size the default profile** — most coursework fits in 1–2 CPU / 4–8 GB; make GPU profiles a deliberate choice, not the default.
- **Idle culling is non-negotiable** — `cull.timeout` of 1 hour reclaims forgotten servers (that's why we set it on every hub).
- **Storage** — home PVC size × enrollment is your Ceph footprint; keep homes small and put datasets on a shared RWX volume.

Requesting a hub without running one yourself: NRP can host custom hubs for courses — [contact the team](https://nrp.ai/contact/). Or run your own, which is exactly what's next.


### 🧠 Quick check

**Which JupyterHub component actually creates the per-user JupyterLab pods?**

- The hub pod, via its spawner
- The proxy pod
- Each user's browser session

<details><summary><b>Reveal answer</b></summary>

**✔ The hub pod, via its spawner**

The hub authenticates users and its spawner (KubeSpawner on z2jh) creates a pod + PVC per user; the proxy just routes traffic to the right server.

</details>

**What makes nbgitpuller links the standard way to hand out course materials?**

- They clone/update a repo into the student's home *without overwriting their edits*
- They force-reset the student's folder to match the repo
- They email students a zip archive

<details><summary><b>Reveal answer</b></summary>

**✔ They clone/update a repo into the student's home *without overwriting their edits***

nbgitpuller does an automatic merge that always preserves student changes — you can re-click the launch link all day without losing work. That's exactly what the buttons on this site do.

</details>

**Why is idle culling non-negotiable on a shared hub?**

- It keeps the hub database small
- It reclaims CPU/RAM/GPU from servers people forgot to stop
- It logs students out for security

<details><summary><b>Reveal answer</b></summary>

**✔ It reclaims CPU/RAM/GPU from servers people forgot to stop**

Forgotten servers hold real resources on shared nodes. The 1-hour cull timeout is why NRP requires it on every hub — you'll set it in your own values file next episode.

</details>

**"Each user server is just a pod." What does that buy you as a hub operator?**

- Everything from this morning applies — images, resource limits, tolerations/affinity, PVCs
- Nothing — hub pods are a special Kubernetes resource type
- Users can kubectl into each other's servers

<details><summary><b>Reveal answer</b></summary>

**✔ Everything from this morning applies — images, resource limits, tolerations/affinity, PVCs**

That's the deep idea of z2jh: the spawner writes an ordinary pod spec. Steering students onto reserved GPU nodes uses the exact toleration+affinity pattern from Episode 2; per-user homes are ordinary PVCs.

</details>

**You're provisioning a hub for a 100-student course. What number do you actually size for?**

- Peak concurrent servers — typically ~40% of enrollment outside deadline nights
- Total enrollment, one server each
- One server per teaching assistant

<details><summary><b>Reveal answer</b></summary>

**✔ Peak concurrent servers — typically ~40% of enrollment outside deadline nights**

Concurrency, not enrollment. With idle culling on, a 100-student course rarely exceeds ~40 simultaneous servers — provision for that (plus deadline-night headroom), and let the cluster autoscale absorb spikes.

</details>


---

## ✅ Check your work

Verifies the state of your resources on the cluster — rerun any time.


In [ ]:
bash check.sh 5
